In [ ]:
# CELL 1 - mount + installs  (~1-2 min)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
os.system(f"{sys.executable} -m pip -q install hnswlib sentence-transformers rapidfuzz pyarabic")
print("ready")


In [ ]:
# ============================================================
#  PART C - WHY DID 2 OF 10 QUERIES NOT MATCH?
#  Hypothesis: the agent REFINED those queries before retrieving.
# ============================================================
import os, json, pickle, hnswlib
from sentence_transformers import SentenceTransformer

ROOT = "/content/drive/MyDrive"
IDX  = f"{ROOT}/Phase1_Project/index_rebuild_base_verses_only/index"
P2   = f"{ROOT}/Phase2_Project/Laiba_integration_output/test_query_results.json"
P3   = f"{ROOT}/Phase3_Project/guardrail_output/phase3_guardrail_results.json"

model = SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1")
entries = pickle.load(open(f"{IDX}/entries.pkl", "rb"))
ix = hnswlib.Index(space="cosine", dim=768)
ix.load_index(f"{IDX}/verses.hnsw")
ix.set_ef(256)
print(f"index: {len(entries):,} entries\n")

def top(q, k=5):
    e = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    lab, dist = ix.knn_query(e, k=k)
    return [(entries[i]["verse_key"], float(1.0 - d)) for i, d in zip(lab[0], dist[0])]

print("#" * 70)
print("# C1 - PHASE 2: original query vs the query actually sent to retrieval")
print("#" * 70)
if os.path.exists(P2):
    for i, d in enumerate(json.load(open(P2, encoding="utf-8")), 1):
        q, fq = d.get("query", ""), d.get("final_query", "")
        rec = d.get("sufficiency_score")
        same = "SAME" if q == fq else "REFINED"
        print(f"\n{i:>2}. [{same}] rounds={d.get('rounds')}  stop={d.get('stopped_reason')}")
        print(f"    original: {q}")
        print(f"    final   : {fq}")
        print(f"    recorded top-sim: {rec}")
        for lbl, qq in (("original", q), ("final", fq)):
            if not qq:
                continue
            t = top(qq, 1)[0]
            hit = "<== MATCHES RECORDED" if rec is not None and abs(t[1] - rec) < 1e-5 else ""
            print(f"    probe {lbl:<8s}: {t[0]:<8s} {t[1]:.10f}  {hit}")
            if lbl == "original" and qq == fq:
                break
else:
    print("  MISSING")

print()
print("#" * 70)
print("# C2 - PHASE 3: do the SAVED context verses come from this same index?")
print("#" * 70)
if os.path.exists(P3):
    for i, d in enumerate(json.load(open(P3, encoding="utf-8")), 1):
        q = d.get("query", "")
        ctx = d.get("context") or []
        rec_keys = [c.get("verse_key") for c in ctx if isinstance(c, dict)]
        rec_sims = [c.get("similarity") for c in ctx if isinstance(c, dict)]
        mine = top(q, 5)
        mine_keys = [k for k, _ in mine]
        print(f"\n{i:>2}. {q[:45]}")
        print(f"    saved  : {rec_keys}")
        print(f"    my top5: {mine_keys}")
        print(f"    saved top-sim {rec_sims[0]:.10f}   mine {mine[0][1]:.10f}   "
              f"{'MATCH' if abs(rec_sims[0]-mine[0][1]) < 1e-5 else 'DIFFERENT'}")
        # are the saved verses even reachable in this index?
        deep = top(q, 300)
        rank = {k: r for r, (k, _) in enumerate(deep, 1)}
        print(f"    saved verses' rank for the ORIGINAL query: "
              f"{[(k, rank.get(k, '>300')) for k in rec_keys]}")
        allv = {e["verse_key"] for e in entries}
        missing = [k for k in rec_keys if k not in allv]
        if missing:
            print(f"    !! NOT IN THIS INDEX AT ALL: {missing}")
else:
    print("  MISSING")

print("\n" + "=" * 70)
print("PART C DONE")
print("=" * 70)


In [ ]:
# ============================================================
#  SECTION B - DOES THE NEW F1/F2 CALIBRATION ACTUALLY WORK?
# ============================================================
import os, sys, json, re, shutil, statistics as st

ROOT = "/content/drive/MyDrive"
SRC  = f"{ROOT}/Phase3_Project/guardrail_output/src"
WORK = "/content/g_src"

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK)
sys.path.insert(0, WORK)
print("copied", len(os.listdir(WORK)), "files to", WORK)

import importlib, subprocess

# pip name != import name for a few packages
PIPNAME = {"sklearn": "scikit-learn", "cv2": "opencv-python", "yaml": "PyYAML",
           "Levenshtein": "python-Levenshtein", "attr": "attrs", "PIL": "Pillow"}

def pip_install(mod):
    name = PIPNAME.get(mod, mod)
    print(f"   installing {name} ...")
    r = subprocess.run([sys.executable, "-m", "pip", "-q", "install", name],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   pip failed for {name}: {r.stderr.strip()[:200]}")
        return False
    importlib.invalidate_caches()
    return True

f1 = f2 = None
tried = set()
for attempt in range(10):
    try:
        f1 = importlib.import_module("f1_calibrate_guardrail")
        f2 = importlib.import_module("f2_build_calibration_dataset")
        print("imported f1 and f2 OK")
        break
    except ModuleNotFoundError as e:
        miss = e.name
        if miss in tried:
            print(f"COULD NOT IMPORT: {miss} still missing after install - giving up")
            break
        tried.add(miss)
        print(f"missing dependency: {miss}")
        if not pip_install(miss):
            break
    except Exception as e:
        print("COULD NOT IMPORT f1/f2:", type(e).__name__, e)
        break
else:
    print("gave up after 10 attempts")
if tried:
    print("auto-installed:", sorted(tried))

QRCD = None
for c in [f"{ROOT}/Phase4_Project/data/qrcd_flat.json",
          f"{ROOT}/Phase2_Project/Roma_output/data/qrcd_flat.json"]:
    if os.path.exists(c):
        QRCD = c; break
print("qrcd:", QRCD)

# ---------------------------------------------------------------- B1
print("\n" + "#" * 70)
print("# B1 - CALIBRATE ON QRCD DISTRACTOR-SWAPS (what f1+f2 are designed to do)")
print("#" * 70)
cal_tol = None
if f1 and f2 and QRCD:
    try:
        recs = f2.load_qrcd(QRCD)
        examples = f2.build_calibration_set(recs, n_pairs=40, seed=42)
        cal_tol, cal_acc, scored = f1.calibrate_max_acceptable_mismatch(examples)
        g = [s["mismatch_score"] for s in scored if s["label"] == "grounded"]
        h = [s["mismatch_score"] for s in scored if s["label"] == "hallucinated"]
        print(f"\n  grounded    mismatch: mean {st.mean(g):.3f}  min {min(g):.3f}  max {max(g):.3f}")
        print(f"  hallucinated mismatch: mean {st.mean(h):.3f}  min {min(h):.3f}  max {max(h):.3f}")
        print(f"  overlap between the two classes: "
              f"{'NONE - trivially separable' if max(g) < min(h) else 'yes, they overlap'}")
    except Exception as e:
        print("  FAILED:", type(e).__name__, e)
else:
    print("  skipped (missing f1/f2/qrcd)")

# ---------------------------------------------------------------- B2
print("\n" + "#" * 70)
print("# B2 - DOES THAT TOLERANCE TRANSFER TO THE 10 REAL RESPONSES?")
print("#" * 70)
RES = f"{ROOT}/Phase3_Project/guardrail_output/phase3_guardrail_results.json"
if f1 and os.path.exists(RES):
    real = json.load(open(RES, encoding="utf-8"))
    rows = []
    for d in real:
        ctx = d.get("context") or []
        inctx = {c.get("verse_key") for c in ctx if isinstance(c, dict)}
        cited = set(re.findall(r"\[(\d+:\d+)\]", str(d.get("final_response", ""))))
        if not cited:
            label = "no_citations"
        elif cited <= inctx:
            label = "grounded"
        else:
            label = "hallucinated"
        ex = {"query": d.get("query", "?"), "context": ctx,
              "response_text": d.get("final_response", ""), "label": label}
        try:
            m = f1.compute_mismatch_for_example(ex)
        except Exception as e:
            m = None
            print("   mismatch failed for", ex["query"][:30], type(e).__name__, e)
        rows.append({"q": ex["query"], "label": label, "recomputed": m,
                     "recorded": d.get("final_mismatch_score"),
                     "verified": d.get("verified"),
                     "n_cited": len(cited), "n_ungrounded": len(cited - inctx)})

    print("\n  query                          label         recomputed  recorded   diff")
    for r in rows:
        rc, rd = r["recomputed"], r["recorded"]
        diff = f"{abs(rc-rd):.4f}" if (rc is not None and rd is not None) else "  -  "
        rcs = f"{rc:.4f}" if rc is not None else " fail "
        print(f"  {r['q'][:28]:<30s} {r['label']:<13s} {rcs:>10s}  {rd:.4f}  {diff}")

    ok = [r for r in rows if r["recomputed"] is not None and r["recorded"] is not None]
    if ok:
        mx = max(abs(r["recomputed"] - r["recorded"]) for r in ok)
        print(f"\n  reproduction check: max diff {mx:.2e} "
              f"({'reproduces the saved run' if mx < 1e-6 else 'DOES NOT reproduce'})")

    ev = [r for r in rows if r["label"] in ("grounded", "hallucinated") and r["recomputed"] is not None]
    def score(tol):
        c = sum(1 for r in ev if (r["recomputed"] <= tol) == (r["label"] == "grounded"))
        return c / len(ev) if ev else 0.0
    print(f"\n  on the {len(ev)} real responses with citations:")
    print(f"    tolerance 0.50 (what the run used)      -> accuracy {score(0.5):.1%}")
    if cal_tol is not None:
        print(f"    tolerance {cal_tol:.2f} (calibrated on QRCD)      -> accuracy {score(cal_tol):.1%}")
    bt = max([x/100 for x in range(101)], key=score)
    print(f"    tolerance {bt:.2f} (best possible on these 10) -> accuracy {score(bt):.1%}")
    ng = sum(1 for r in ev if r["label"] == "grounded")
    print(f"    always-say-hallucinated baseline        -> accuracy {1-ng/len(ev):.1%}" if ev else "")
else:
    print("  skipped")

print("\n" + "=" * 70)
print("SECTION B DONE")
print("=" * 70)
